In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing import image
import numpy as np
import cv2
import spektral
from spektral.utils import graph
from scipy.sparse import coo_matrix
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Model
import matplotlib.pyplot as plt

# Load and preprocess the MRI image
image_path = r"D:\Dataset\Brain_Tumor\four_class\Glioma\Te-gl_0011.jpg"
img = image.load_img(image_path, target_size=(224, 224))
img_array = image.img_to_array(img)
img_array = img_array / 255.0  # Normalize pixel values
img_array = np.expand_dims(img_array, axis=0)  # Add batch dimension

# Use pre-trained VGG16 to extract features from the MRI image (Euclidean features)
base_model = VGG16(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
model = Model(inputs=base_model.input, outputs=base_model.output)
euclidean_features = model.predict(img_array)

# Create a grid graph based on image size (non-Euclidean features)
image_height, image_width, _ = img_array.shape
nodes = image_height * image_width  # Number of pixels (nodes)
adjacency_matrix = np.zeros((nodes, nodes))

# Define adjacency (4-connected neighbors: up, down, left, right)
for i in range(image_height):
    for j in range(image_width):
        node_id = i * image_width + j
        if i > 0:
            adjacency_matrix[node_id, (i - 1) * image_width + j] = 1  # Above
        if i < image_height - 1:
            adjacency_matrix[node_id, (i + 1) * image_width + j] = 1  # Below
        if j > 0:
            adjacency_matrix[node_id, i * image_width + (j - 1)] = 1  # Left
        if j < image_width - 1:
            adjacency_matrix[node_id, i * image_width + (j + 1)] = 1  # Right

# Convert the adjacency matrix to a sparse matrix format
adj_matrix = coo_matrix(adjacency_matrix)

# Reshape the image into a node feature matrix (using pixel intensity or color features)
node_features = img_array.reshape((-1, 3))  # Flatten the image, use RGB features as node features

# Assuming you already have a trained GCN model (gcn_model), apply it
# In this case, we simulate a trained GCN model by assuming the 'gcn_model' is defined
# and outputting some result (replace it with actual inference code if you have a trained model)
gcn_output = np.random.rand(image_height * image_width, 1)  # Simulating GCN output

# Visualize the Euclidean features (edges, textures)
plt.subplot(1, 2, 1)
plt.imshow(euclidean_features[0, :, :, 0], cmap='hot')  # Visualize one channel (e.g., edge)
plt.title("Euclidean Features (CNN)")

# Visualize Non-Euclidean features (GCN output)
plt.subplot(1, 2, 2)
plt.imshow(gcn_output.reshape(image_height, image_width), cmap='hot')  # Visualize GCN output
plt.title("Non-Euclidean Features (GCN)")

plt.show()


In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing import image
import numpy as np
from scipy.sparse import coo_matrix
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Model
import matplotlib.pyplot as plt

# Load and preprocess the MRI image
image_path = r"D:\Dataset\Brain_Tumor\four_class\Glioma\Te-gl_0011.jpg"
img = image.load_img(image_path, target_size=(224, 224))
img_array = image.img_to_array(img)
img_array = img_array / 255.0  # Normalize pixel values
img_array = np.expand_dims(img_array, axis=0)  # Add batch dimension

# Extract image shape without batch dimension
image_height, image_width, _ = img_array[0].shape

# Use pre-trained VGG16 to extract features from the MRI image (Euclidean features)
base_model = VGG16(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
model = Model(inputs=base_model.input, outputs=base_model.output)
cnn_feature_maps = model.predict(img_array)

# Use Global Average Pooling to reduce feature maps to 2D
from tensorflow.keras.layers import GlobalAveragePooling2D
pooled_features = GlobalAveragePooling2D()(cnn_feature_maps)
pooled_features = pooled_features.numpy()

# Create a grid graph based on image size (non-Euclidean features)
nodes = image_height * image_width  # Number of pixels (nodes)
adjacency_matrix = np.zeros((nodes, nodes))

# Define adjacency (4-connected neighbors: up, down, left, right)
for i in range(image_height):
    for j in range(image_width):
        node_id = i * image_width + j
        if i > 0:
            adjacency_matrix[node_id, (i - 1) * image_width + j] = 1  # Above
        if i < image_height - 1:
            adjacency_matrix[node_id, (i + 1) * image_width + j] = 1  # Below
        if j > 0:
            adjacency_matrix[node_id, i * image_width + (j - 1)] = 1  # Left
        if j < image_width - 1:
            adjacency_matrix[node_id, i * image_width + (j + 1)] = 1  # Right

# Convert the adjacency matrix to a sparse matrix format
adj_matrix = coo_matrix(adjacency_matrix)

# Reshape the image into a node feature matrix (using pixel intensity or color features)
node_features = img_array.reshape((-1, 3))  # Flatten the image, use RGB features as node features

# Assuming you already have a trained GCN model (gcn_model), apply it
# Load your trained GCN model (adjust the path accordingly)
gcn_model = tf.keras.models.load_model('path_to_your_trained_gcn_model') 

# Perform inference with the graph data (adjacency matrix and node features)
gcn_output = gcn_model.predict([node_features, adj_matrix])

# Visualize the Euclidean features (refined CNN)
plt.subplot(1, 2, 1)
plt.imshow(pooled_features, cmap='hot')
plt.title("Refined CNN Features")

# Visualize Non-Euclidean features (GCN output)
plt.subplot(1, 2, 2)
plt.imshow(gcn_output.reshape(image_height, image_width), cmap='hot')  # Visualize GCN output
plt.title("Non-Euclidean Features (GCN)")

plt.show()


In [ ]:
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import cv2
from tensorflow.keras.preprocessing import image

# Load MRI Image
image_path = 'D:/Dataset/Brain_Tumor/four_class/Glioma/Te-gl_0011.jpg'  # Update the path if needed
img = image.load_img(image_path, target_size=(224, 224))  # Resize for simplicity
img_array = image.img_to_array(img)
img_array = img_array / 255.0  # Normalize the image

# Convert image to grayscale for simplicity (optional)
gray_img = cv2.cvtColor(img_array, cv2.COLOR_RGB2GRAY)

# Create grid graph (nodes connected based on adjacency in the image)
image_height, image_width = gray_img.shape
nodes = image_height * image_width
adjacency_matrix = np.zeros((nodes, nodes))

# Define adjacency (4-connected neighbors: up, down, left, right)
for i in range(image_height):
    for j in range(image_width):
        node_id = i * image_width + j
        if i > 0:
            adjacency_matrix[node_id, (i - 1) * image_width + j] = 1  # Above
        if i < image_height - 1:
            adjacency_matrix[node_id, (i + 1) * image_width + j] = 1  # Below
        if j > 0:
            adjacency_matrix[node_id, i * image_width + (j - 1)] = 1  # Left
        if j < image_width - 1:
            adjacency_matrix[node_id, i * image_width + (j + 1)] = 1  # Right

# Create graph using NetworkX (correct function call)
G = nx.from_numpy_array(adjacency_matrix)

# Visualize the graph (with node positions in a grid)
pos = {i: (i % image_width, i // image_width) for i in range(nodes)}

# Highlight a subset of nodes (for example, based on intensity or features)
highlight_nodes = np.random.choice(nodes, 50, replace=False)  # Randomly highlight 50 nodes (can be based on features)
node_colors = ['orange' if i in highlight_nodes else 'lightblue' for i in range(nodes)]

# Plot the graph
plt.figure(figsize=(10, 8))
nx.draw(G, pos, node_size=10, node_color=node_colors, with_labels=False, edge_color='gray', alpha=0.6)
plt.title("Graph Representation of MRI Image (Nodes and Edges)")
plt.show()


In [ ]:
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import cv2
from tensorflow.keras.preprocessing import image

# Load MRI Image
image_path = 'D:/Dataset/Brain_Tumor/four_class/Glioma/Te-gl_0011.jpg'  # Path to your MRI image
img = image.load_img(image_path)  # Load image without resizing to keep original size
img_array = image.img_to_array(img)
img_array = img_array / 255.0  # Normalize the image

# Convert image to grayscale for simplicity (optional)
gray_img = cv2.cvtColor(img_array, cv2.COLOR_RGB2GRAY)

# Create grid graph (nodes connected based on adjacency in the image)
image_height, image_width = gray_img.shape
nodes = image_height * image_width
adjacency_matrix = np.zeros((nodes, nodes))

# Define adjacency (4-connected neighbors: up, down, left, right)
for i in range(image_height):
    for j in range(image_width):
        node_id = i * image_width + j
        if i > 0:
            adjacency_matrix[node_id, (i - 1) * image_width + j] = 1  # Above
        if i < image_height - 1:
            adjacency_matrix[node_id, (i + 1) * image_width + j] = 1  # Below
        if j > 0:
            adjacency_matrix[node_id, i * image_width + (j - 1)] = 1  # Left
        if j < image_width - 1:
            adjacency_matrix[node_id, i * image_width + (j + 1)] = 1  # Right

# Create graph using NetworkX (correct function call)
G = nx.from_numpy_array(adjacency_matrix)

# Visualize the graph (with node positions in a grid)
pos = {i: (i % image_width, i // image_width) for i in range(nodes)}

# Highlight a subset of nodes (for example, based on intensity or features)
highlight_nodes = np.random.choice(nodes, 50, replace=False)  # Randomly highlight 50 nodes (can be based on features)
node_colors = ['black' if i in highlight_nodes else 'white' for i in range(nodes)]  # Black for highlighted, white for others

# Adjusting figure size and layout for better visibility
plt.figure(figsize=(20, 20))  # Maintain larger figure size to match the original image size

# Draw graph with larger nodes and adjusted layout
nx.draw(G, pos, node_size=10, node_color=node_colors, with_labels=False, edge_color='gray', alpha=0.6, width=0.5)

# Title and display
plt.title("Graph Representation of MRI Image (Nodes and Edges)")
plt.show()


In [ ]:
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import cv2
from tensorflow.keras.preprocessing import image
from scipy.sparse import lil_matrix

# Load MRI Image
image_path = 'D:/Dataset/Brain_Tumor/four_class/Glioma/Te-gl_0011.jpg'  # Path to your MRI image
img = image.load_img(image_path)  # Load image without resizing to keep original size
img_array = image.img_to_array(img)
img_array = img_array / 255.0  # Normalize the image

# Convert image to grayscale for simplicity (optional)
gray_img = cv2.cvtColor(img_array, cv2.COLOR_RGB2GRAY)

# Downsample the image to reduce the number of pixels
downsample_factor = 10  # For example, downsample by a factor of 10
small_gray_img = gray_img[::downsample_factor, ::downsample_factor]

# Get the dimensions of the downsampled image
image_height, image_width = small_gray_img.shape
nodes = image_height * image_width

# Create sparse adjacency matrix (lil_matrix format is memory efficient)
adjacency_matrix = lil_matrix((nodes, nodes))

# Define adjacency (4-connected neighbors: up, down, left, right)
for i in range(image_height):
    for j in range(image_width):
        node_id = i * image_width + j
        if i > 0:  # Above
            adjacency_matrix[node_id, (i - 1) * image_width + j] = 1
        if i < image_height - 1:  # Below
            adjacency_matrix[node_id, (i + 1) * image_width + j] = 1
        if j > 0:  # Left
            adjacency_matrix[node_id, i * image_width + (j - 1)] = 1
        if j < image_width - 1:  # Right
            adjacency_matrix[node_id, i * image_width + (j + 1)] = 1

# Create graph using NetworkX from the sparse adjacency matrix
G = nx.from_scipy_sparse_matrix(adjacency_matrix)

# Visualize the graph (with node positions in a grid)
pos = {i: (i % image_width, i // image_width) for i in range(nodes)}

# Highlight a subset of nodes (for example, based on intensity or features)
highlight_nodes = np.random.choice(nodes, 50, replace=False)  # Randomly highlight 50 nodes (can be based on features)
node_colors = ['black' if i in highlight_nodes else 'white' for i in range(nodes)]  # Black for highlighted, white for others

# Adjusting figure size and layout for better visibility
plt.figure(figsize=(15, 15))  # Increase figure size for better visibility

# Draw graph with larger nodes and adjusted layout
nx.draw(G, pos, node_size=10, node_color=node_colors, with_labels=False, edge_color='gray', alpha=0.6, width=0.5)

# Title and display
plt.title("Graph Representation of MRI Image (Nodes and Edges)")
plt.show()


In [ ]:
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import cv2
from tensorflow.keras.preprocessing import image
from scipy.sparse import lil_matrix

# Load MRI Image
image_path = 'D:/Dataset/Brain_Tumor/four_class/Glioma/Te-gl_0011.jpg'  # Path to your MRI image
img = image.load_img(image_path)  # Load image without resizing to keep original size
img_array = image.img_to_array(img)
img_array = img_array / 255.0  # Normalize the image

# Convert image to grayscale for simplicity (optional)
gray_img = cv2.cvtColor(img_array, cv2.COLOR_RGB2GRAY)

# Downsample the image to reduce the number of pixels
downsample_factor = 10  # For example, downsample by a factor of 10
small_gray_img = gray_img[::downsample_factor, ::downsample_factor]

# Get the dimensions of the downsampled image
image_height, image_width = small_gray_img.shape
nodes = image_height * image_width

# Create sparse adjacency matrix (lil_matrix format is memory efficient)
adjacency_matrix = lil_matrix((nodes, nodes))

# Define adjacency (4-connected neighbors: up, down, left, right)
for i in range(image_height):
    for j in range(image_width):
        node_id = i * image_width + j
        if i > 0:  # Above
            adjacency_matrix[node_id, (i - 1) * image_width + j] = 1
        if i < image_height - 1:  # Below
            adjacency_matrix[node_id, (i + 1) * image_width + j] = 1
        if j > 0:  # Left
            adjacency_matrix[node_id, i * image_width + (j - 1)] = 1
        if j < image_width - 1:  # Right
            adjacency_matrix[node_id, i * image_width + (j + 1)] = 1

# Create graph using NetworkX from the sparse adjacency matrix
G = nx.Graph()  # Initialize the graph object
for i in range(nodes):
    for j in adjacency_matrix[i].indices:
        G.add_edge(i, j)  # Add an edge between nodes i and j

# Visualize the graph (with node positions in a grid)
pos = {i: (i % image_width, i // image_width) for i in range(nodes)}

# Highlight a subset of nodes (for example, based on intensity or features)
highlight_nodes = np.random.choice(nodes, 50, replace=False)  # Randomly highlight 50 nodes (can be based on features)
node_colors = ['black' if i in highlight_nodes else 'white' for i in range(nodes)]  # Black for highlighted, white for others

# Adjusting figure size and layout for better visibility
plt.figure(figsize=(15, 15))  # Increase figure size for better visibility

# Draw graph with larger nodes and adjusted layout
nx.draw(G, pos, node_size=10, node_color=node_colors, with_labels=False, edge_color='gray', alpha=0.6, width=0.5)

# Title and display
plt.title("Graph Representation of MRI Image (Nodes and Edges)")
plt.show()


In [ ]:
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import cv2
from tensorflow.keras.preprocessing import image
from scipy.sparse import lil_matrix

# Load MRI Image
image_path = 'D:/Dataset/Brain_Tumor/four_class/Glioma/Te-gl_0011.jpg'  # Path to your MRI image
img = image.load_img(image_path)  # Load image without resizing to keep original size
img_array = image.img_to_array(img)
img_array = img_array / 255.0  # Normalize the image

# Convert image to grayscale for simplicity (optional)
gray_img = cv2.cvtColor(img_array, cv2.COLOR_RGB2GRAY)

# Downsample the image to reduce the number of pixels
downsample_factor = 10  # For example, downsample by a factor of 10
small_gray_img = gray_img[::downsample_factor, ::downsample_factor]

# Get the dimensions of the downsampled image
image_height, image_width = small_gray_img.shape
nodes = image_height * image_width

# Create sparse adjacency matrix (lil_matrix format is memory efficient)
adjacency_matrix = lil_matrix((nodes, nodes))

# Define adjacency (4-connected neighbors: up, down, left, right)
for i in range(image_height):
    for j in range(image_width):
        node_id = i * image_width + j
        if i > 0:  # Above
            adjacency_matrix[node_id, (i - 1) * image_width + j] = 1
        if i < image_height - 1:  # Below
            adjacency_matrix[node_id, (i + 1) * image_width + j] = 1
        if j > 0:  # Left
            adjacency_matrix[node_id, i * image_width + (j - 1)] = 1
        if j < image_width - 1:  # Right
            adjacency_matrix[node_id, i * image_width + (j + 1)] = 1

# Create graph using NetworkX from the sparse adjacency matrix
G = nx.Graph()  # Initialize the graph object

# Iterate over the rows in the sparse matrix to add edges
for i in range(nodes):
    # Access the neighbors for node i using adjacency_matrix[i].data and adjacency_matrix[i].rows
    for j, weight in zip(adjacency_matrix[i].rows[0], adjacency_matrix[i].data[0]):
        if weight != 0:
            G.add_edge(i, j)  # Add an edge between nodes i and j

# Visualize the graph (with node positions in a grid)
pos = {i: (i % image_width, i // image_width) for i in range(nodes)}

# Highlight a subset of nodes (for example, based on intensity or features)
highlight_nodes = np.random.choice(nodes, 50, replace=False)  # Randomly highlight 50 nodes (can be based on features)
node_colors = ['black' if i in highlight_nodes else 'white' for i in range(nodes)]  # Black for highlighted, white for others

# Adjusting figure size and layout for better visibility
plt.figure(figsize=(15, 15))  # Increase figure size for better visibility

# Draw graph with larger nodes and adjusted layout
nx.draw(G, pos, node_size=10, node_color=node_colors, with_labels=False, edge_color='gray', alpha=0.6, width=0.5)

# Title and display
plt.title("Graph Representation of MRI Image (Nodes and Edges)")
plt.show()


In [ ]:
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import cv2
from tensorflow.keras.preprocessing import image
from scipy.sparse import lil_matrix

# Load MRI Image
image_path = 'D:/Dataset/Brain_Tumor/four_class/Glioma/Te-gl_0011.jpg'  # Path to your MRI image
img = image.load_img(image_path)  # Load image without resizing to keep original size
img_array = image.img_to_array(img)
img_array = img_array / 255.0  # Normalize the image

# Convert image to grayscale for simplicity
gray_img = cv2.cvtColor(img_array, cv2.COLOR_RGB2GRAY)

# Downsample the image to reduce the number of pixels
downsample_factor = 10  # Downsample by a factor of 10
small_gray_img = gray_img[::downsample_factor, ::downsample_factor]

# Edge detection using Canny
edges = cv2.Canny((small_gray_img * 255).astype(np.uint8), threshold1=100, threshold2=200)

# Get the dimensions of the downsampled image
image_height, image_width = small_gray_img.shape
nodes = image_height * image_width

# Create sparse adjacency matrix (lil_matrix format is memory efficient)
adjacency_matrix = lil_matrix((nodes, nodes))

# Define adjacency (4-connected neighbors: up, down, left, right)
for i in range(image_height):
    for j in range(image_width):
        node_id = i * image_width + j
        if i > 0:  # Above
            adjacency_matrix[node_id, (i - 1) * image_width + j] = 1
        if i < image_height - 1:  # Below
            adjacency_matrix[node_id, (i + 1) * image_width + j] = 1
        if j > 0:  # Left
            adjacency_matrix[node_id, i * image_width + (j - 1)] = 1
        if j < image_width - 1:  # Right
            adjacency_matrix[node_id, i * image_width + (j + 1)] = 1

# Create graph using NetworkX from the sparse adjacency matrix
G = nx.Graph()  # Initialize the graph object

# Iterate over the rows in the sparse matrix to add edges
for i in range(nodes):
    for j, weight in zip(adjacency_matrix[i].rows[0], adjacency_matrix[i].data[0]):
        if weight != 0:
            G.add_edge(i, j)  # Add an edge between nodes i and j

# Visualize the graph (with node positions in a grid)
pos = {i: (i % image_width, i // image_width) for i in range(nodes)}

# Highlight the nodes based on edge detection (Canny edges)
highlight_nodes = []
for i in range(image_height):
    for j in range(image_width):
        if edges[i, j] == 255:  # Canny edge detected pixel
            node_id = i * image_width + j
            highlight_nodes.append(node_id)

# Set node colors: Black for highlighted nodes (edges) and white for others
node_colors = ['black' if i in highlight_nodes else 'white' for i in range(nodes)]

# Adjusting figure size and layout for better visibility
plt.figure(figsize=(15, 15))  # Increase figure size for better visibility

# Draw graph with larger nodes and adjusted layout
nx.draw(G, pos, node_size=10, node_color=node_colors, with_labels=False, edge_color='gray', alpha=0.6, width=0.5)

# Title and display
plt.title("Graph Representation of MRI Image (Edges Highlighted)")
plt.show()


In [ ]:
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import cv2
from tensorflow.keras.preprocessing import image
from scipy.sparse import lil_matrix

# Load MRI Image
image_path = 'D:/Dataset/Brain_Tumor/four_class/Glioma/Te-gl_0011.jpg'  # Path to your MRI image
img = image.load_img(image_path)  # Load image without resizing to keep original size
img_array = image.img_to_array(img)
img_array = img_array / 255.0  # Normalize the image

# Convert image to grayscale for simplicity
gray_img = cv2.cvtColor(img_array, cv2.COLOR_RGB2GRAY)

# Downsample the image to reduce the number of pixels
downsample_factor = 10  # Downsample by a factor of 10
small_gray_img = gray_img[::downsample_factor, ::downsample_factor]

# Edge detection using Canny
edges = cv2.Canny((small_gray_img * 255).astype(np.uint8), threshold1=100, threshold2=200)

# Get the dimensions of the downsampled image
image_height, image_width = small_gray_img.shape
nodes = image_height * image_width

# Create sparse adjacency matrix (lil_matrix format is memory efficient)
adjacency_matrix = lil_matrix((nodes, nodes))

# Define adjacency (4-connected neighbors: up, down, left, right)
for i in range(image_height):
    for j in range(image_width):
        node_id = i * image_width + j
        if i > 0:  # Above
            adjacency_matrix[node_id, (i - 1) * image_width + j] = 1
        if i < image_height - 1:  # Below
            adjacency_matrix[node_id, (i + 1) * image_width + j] = 1
        if j > 0:  # Left
            adjacency_matrix[node_id, i * image_width + (j - 1)] = 1
        if j < image_width - 1:  # Right
            adjacency_matrix[node_id, i * image_width + (j + 1)] = 1

# Create graph using NetworkX from the sparse adjacency matrix
G = nx.Graph()  # Initialize the graph object

# Iterate over the rows in the sparse matrix to add edges
for i in range(nodes):
    for j, weight in zip(adjacency_matrix[i].rows[0], adjacency_matrix[i].data[0]):
        if weight != 0:
            G.add_edge(i, j)  # Add an edge between nodes i and j

# Visualize the graph (with node positions in a grid)
pos = {i: (i % image_width, i // image_width) for i in range(nodes)}

# Highlight the nodes based on edge detection (Canny edges)
highlight_nodes = []
for i in range(image_height):
    for j in range(image_width):
        if edges[i, j] == 255:  # Canny edge detected pixel
            node_id = i * image_width + j
            highlight_nodes.append(node_id)

# Set node colors: Black for highlighted nodes (edges) and white for others
node_colors = ['black' if i in highlight_nodes else 'white' for i in range(nodes)]

# Adjusting figure size and layout for better visibility
plt.figure(figsize=(20, 20))  # Increase figure size for better visibility

# Draw graph with larger nodes and adjusted layout
nx.draw(G, pos, node_size=50, node_color=node_colors, with_labels=False, edge_color='gray', alpha=0.6, width=0.5)

# Title and display
plt.title("Graph Representation of MRI Image (Edges Highlighted)")
plt.show()


In [ ]:
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import cv2
from tensorflow.keras.preprocessing import image
from scipy.sparse import lil_matrix

# Load MRI Image
image_path = 'D:/Dataset/Brain_Tumor/four_class/Glioma/Te-gl_0011.jpg'  # Path to your MRI image
img = image.load_img(image_path)  # Load image without resizing to keep original size
img_array = image.img_to_array(img)
img_array = img_array / 255.0  # Normalize the image

# Convert image to grayscale for simplicity
gray_img = cv2.cvtColor(img_array, cv2.COLOR_RGB2GRAY)

# Downsample the image to reduce the number of pixels
downsample_factor = 10  # Downsample by a factor of 10
small_gray_img = gray_img[::downsample_factor, ::downsample_factor]

# Edge detection using Canny
edges = cv2.Canny((small_gray_img * 255).astype(np.uint8), threshold1=100, threshold2=200)

# Get the dimensions of the downsampled image
image_height, image_width = small_gray_img.shape
nodes = image_height * image_width

# Create sparse adjacency matrix (lil_matrix format is memory efficient)
adjacency_matrix = lil_matrix((nodes, nodes))

# Define adjacency (4-connected neighbors: up, down, left, right)
for i in range(image_height):
    for j in range(image_width):
        node_id = i * image_width + j
        if i > 0:  # Above
            adjacency_matrix[node_id, (i - 1) * image_width + j] = 1
        if i < image_height - 1:  # Below
            adjacency_matrix[node_id, (i + 1) * image_width + j] = 1
        if j > 0:  # Left
            adjacency_matrix[node_id, i * image_width + (j - 1)] = 1
        if j < image_width - 1:  # Right
            adjacency_matrix[node_id, i * image_width + (j + 1)] = 1

# Create graph using NetworkX from the sparse adjacency matrix
G = nx.Graph()  # Initialize the graph object

# Iterate over the rows in the sparse matrix to add edges
for i in range(nodes):
    for j, weight in zip(adjacency_matrix[i].rows[0], adjacency_matrix[i].data[0]):
        if weight != 0:
            G.add_edge(i, j)  # Add an edge between nodes i and j

# Visualize the graph (with node positions in a grid)
pos = {i: (i % image_width, i // image_width) for i in range(nodes)}

# Highlight the nodes based on edge detection (Canny edges)
highlight_nodes = []
for i in range(image_height):
    for j in range(image_width):
        if edges[i, j] == 255:  # Canny edge detected pixel
            node_id = i * image_width + j
            highlight_nodes.append(node_id)

# Set node colors: Red for Euclidean data (edges) and blue for non-Euclidean
node_colors = []
for i in range(nodes):
    if i in highlight_nodes:
        node_colors.append('red')  # Euclidean data (edge detected)
    else:
        node_colors.append('blue')  # Non-Euclidean data (non-edge detected)

# Adjusting figure size and layout for better visibility
plt.figure(figsize=(15, 15))  # Increase figure size for better visibility

# Draw graph with larger nodes and adjusted layout
nx.draw(G, pos, node_size=10, node_color=node_colors, with_labels=False, edge_color='gray', alpha=0.6, width=0.5)

# Title and display
plt.title("Graph Representation of MRI Image (Euclidean and Non-Euclidean Data Highlighted)")
plt.show()


In [ ]:
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import cv2
from tensorflow.keras.preprocessing import image
from scipy.sparse import lil_matrix

# Load MRI Image
image_path = 'D:/Dataset/Brain_Tumor/four_class/Glioma/Te-gl_0011.jpg'  # Path to your MRI image
img = image.load_img(image_path)  # Load image without resizing to keep original size
img_array = image.img_to_array(img)
img_array = img_array / 255.0  # Normalize the image

# Convert image to grayscale for simplicity
gray_img = cv2.cvtColor(img_array, cv2.COLOR_RGB2GRAY)

# Downsample the image to reduce the number of pixels
downsample_factor = 10  # Downsample by a factor of 10
small_gray_img = gray_img[::downsample_factor, ::downsample_factor]

# Edge detection using Canny
edges = cv2.Canny((small_gray_img * 255).astype(np.uint8), threshold1=100, threshold2=200)

# Get the dimensions of the downsampled image
image_height, image_width = small_gray_img.shape
nodes = image_height * image_width

# Create sparse adjacency matrix (lil_matrix format is memory efficient)
adjacency_matrix = lil_matrix((nodes, nodes))

# Define adjacency (4-connected neighbors: up, down, left, right)
for i in range(image_height):
    for j in range(image_width):
        node_id = i * image_width + j
        if i > 0:  # Above
            adjacency_matrix[node_id, (i - 1) * image_width + j] = 1
        if i < image_height - 1:  # Below
            adjacency_matrix[node_id, (i + 1) * image_width + j] = 1
        if j > 0:  # Left
            adjacency_matrix[node_id, i * image_width + (j - 1)] = 1
        if j < image_width - 1:  # Right
            adjacency_matrix[node_id, i * image_width + (j + 1)] = 1

# Create graph using NetworkX from the sparse adjacency matrix
G = nx.Graph()  # Initialize the graph object

# Iterate over the rows in the sparse matrix to add edges
for i in range(nodes):
    for j, weight in zip(adjacency_matrix[i].rows[0], adjacency_matrix[i].data[0]):
        if weight != 0:
            G.add_edge(i, j)  # Add an edge between nodes i and j

# Visualize the graph (with node positions in a grid)
pos = {i: (i % image_width, i // image_width) for i in range(nodes)}

# Highlight the nodes based on edge detection (Canny edges)
highlight_nodes = []
for i in range(image_height):
    for j in range(image_width):
        if edges[i, j] == 255:  # Canny edge detected pixel
            node_id = i * image_width + j
            highlight_nodes.append(node_id)

# Set node colors: Red for Euclidean data (edges) and blue for non-Euclidean
node_colors = []
for i in range(nodes):
    if i in highlight_nodes:
        node_colors.append('red')  # Euclidean data (edge detected)
    else:
        node_colors.append('green')  # Non-Euclidean data (non-edge detected)

# Adjusting figure size and layout for better visibility
plt.figure(figsize=(20, 20))  # Increased figure size

# Draw graph with much larger nodes and adjusted layout
nx.draw(G, pos, node_size=50, node_color=node_colors, with_labels=False, edge_color='gray', alpha=0.6, width=0.5)

# Title and display
plt.title("Graph Representation of MRI Image (Euclidean and Non-Euclidean Data Highlighted)")
plt.show()
